# Lesson 5 : Knowledge - Foundry IQ (Azure AI Search)

By integrating with Foundry IQ (Azure AI search), you can easily build your agent to use grounded retrieval, based on reliable and up-to-date information - not only web search results, but also business knowledge (Work IQ, SharePoint), enterprise analytical data (Fabric IQ, OneLake), and so on.<br>
With Foundry IQ, you can:

- Quickly implement common enterprise scenarios without having to build the entire solution from scratch - such as, integrating with semantic business knowledge, restricted access (personal e-mails, team documents), etc
- Easily configure advanced grounding capabilities to improve the accuracy and speed of AI-generated responses - such as, hybrid search, vector compression, semantic reranking, etc

In this exercise, we briefly configure Foundry IQ in Microsoft Foundry to ground document on blob storage, and create an agent to interact with this knowledge base in Microsoft Agent Framework.

## 1. Create a blob container with document

First, as preparation, we create a container in Azure Blob storage and upload a document with Azure Portal as follows.

- Create a new storage account resource.
- Create a new container in this storage account. (Set "private" (default setting) as access level.)
- In this repository, there exist a sample document, [sample_retail_document.docx](https://github.com/tsmatsuz/agent-framework-workshop-with-foundry/tree/master/assets/sample_retail_document.docx). Please upload this document in above container.

> Note : Using Azure Data Lake Storage (ADLS) Gen2, you can configure ACL permissions and filter document in search, based on Entra ID user access rights. See [here](https://learn.microsoft.com/en-us/azure/search/search-indexer-access-control-lists-and-role-based-access) for details.

## 2. Deploy an embedding model

We have already deployed a chat model in [Readme.md](./Readme.md).<br>
Please additionally deploy an embedding model (such as, text-embedding-3-small) in Microsoft Foundry.

## 3. Prepare a AI search (Foundry IQ) resource

If you haven't set up a knowledge base in this Foundry project yet, you should create a Azure AI search (Foundry IQ) resource at first as follows.

- Open Foundry Portal.
- Go to "Build" tab.
- Select "Knowledge" menu.
- You can create a new resource by clicking "create new resource" link.

The new AI search (Foundry IQ) resource will be created within the same resource group as the Foundry resource.

After creation, go to AI search (Foundry IQ) resource in Azure Portal, copy url (search endpoint) of this resource, and please fill this url in the following variable.

In [1]:
# ToDo : fill url (e.g, "https://mysearch.search.windows.net")
search_endpoint="[Fill-URL-of-Your-Search-Service]"

## 4. [Optional] Role assignment (permisison) setting to managed identities

**Only when you use managed identity authentication** (instead of default API key authentication) in the following knowledge base setting, you should configure role assignment (IAM) for Azure AI resource and Foundry resource on Azure Portal.

> Note : This setting will be required when your organization restricts API key authentication.

Firstly, enable managed identity (system assigned identity or user assigned identity) on Azure AI search resource as follows.

- Go to AI search (Foundry IQ) resource you have created above on Azure Portal. (You AI search resource will be on the same resource group as the Foundry resource.)
- Select "Security + Networking" - "Identity" menu.
- Assign managed identity (system assigned or user assigned), and save the setting.

Next, add role assignments.<br>
When you assign *role_A* in *resource_A* to the managed identity of *resource_B*, you should perform the following steps :

1. Go to *resource_A* in Azure Portal.
2. Select "Access Control (IAM)" menu.
3. Assign (add role assignment) *role_A* to a managed identity of *resource_B*.

Now let's assign roles as follows.

In AI search (Foundry IQ) resource :
- Assign "Search Index Data Reader" role to the managed idenity of Foundry resource.
- Assign "Search Index Data Reader" role to the managed idenity of Foundry project resource.
- Assign "Search Service Contributor" role to the managed idenity of Foundry resource.
- Assign "Search Service Contributor" role to the managed idenity of Foundry project resource.

In Foundry resource (not Foundry project resource) :
- Assign "Foundry User" role to the managed idenity of AI search resource.
- Assign "Cognitive Services User" role to the managed idenity of AI search resource.

In storage account resource :
- Assign "Storage Blob Data Reader" role to the managed idenity of AI search resource.

> Note : Role assignment in Foundry project resource is not required, because the role assignment is inherited from parent Foundry resource.

## 5. Configure Knowledge base in Foundry IQ

Then, we configure a knowledge base for document retrieval in Foundry IQ on Foundry portal as follows.<br>
Go to knowledge setting in Foundry portal and create a new knowledge base using the configuration below.

- output mode: extractive data (default setting)
- knowledge sources: Azure Blob Storage
    - Connect to above storage account and container
    - Embedding model: (Select an embedding model you have deployed above)
    - Chat completion model: (Select a chat model already deployed)

After setting, make sure to push "save" button.

> Note : The "chat completion model" in knowledge base setting is used for image verbalization and content extraction in document cracking.

With this setting, AI Search start to collect keyword and vector indexes.<br>
Go to AI search (Foundry IQ) resource in Azure Portal, and **make sure that indexer status is successful**. (If it's failed, fix the bug and run indexer again.)

After completion, please fill knowledge base name in the following variable.

In [2]:
# ToDo : fill knowledge base name
knowledge_base_name="[Fill-Knowledge-Base-Name]"

> Note : You can configure knowledge base with Foundry SDK (programming code). See [here](https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-blob) for details.

## 6. Assign role to read search index

As you'll see later, we'll call AI search provider using your credential with ```AzureCliCredential()``` in this example.<br>
So assign "Search Index Data Reader" in AI search resource to yourself as follows :

1. Go to AI search resource in Azure Portal.
2. Select "Access Control (IAM)" menu.
3. Assign (add role assignment) "Search Index Data Reader" to you.

## 7. Build and run an agent working with Foundry IQ

All preparations are now complete.<br>
Let's build your own custom agent using Foundry IQ knowledge base in Microsoft Agent Framework.

Firstly, we create a client as follows.

In [3]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

Build your agent using above knowledge base.<br>
The following ```AzureAISearchContextProvider``` provides 2 modes below. (Use agentic mode in most cases.)

- agentic : Excellent for generic or complex queries, because it provides query planning and multi-hop reasoning.
- semantic : This is fast and good for simple queries, because it provides single-hop lookup.

In [4]:
from agent_framework import Agent
from agent_framework.azure import AzureAISearchContextProvider

search_provider = AzureAISearchContextProvider(
    source_id="search_provider",
    endpoint=search_endpoint,
    credential=AzureCliCredential(),
    mode="agentic",
    knowledge_base_name=knowledge_base_name,
)

agent = Agent(
    name="AgentWithFoundryIQ",
    client=client,
    instructions=(
        "You are a helpful assistant. "
        "Use the provided context to answer questions about company report."
    ),
    context_providers=[search_provider],
)

Now let's ask the following question.<br>
The document [sample_retail_document.docx](https://github.com/tsmatsuz/agent-framework-workshop-with-foundry/tree/master/assets/sample_retail_document.docx) tells that top selling category is "Electronics" and total sales is $50,000.<br>
The agent will then provide the same answer as follows by grounding this information.

In [5]:
from IPython.display import Markdown, display

result = await agent.run("According to the retail report, what is the top-selling category ?")
display(Markdown(result.text))

The top-selling category is **Electronics**.